# 第06章：数学运算与类型系统

## 前置知识
- 第01-05章

## 学习目标
- 掌握 Triton 提供的数学函数
- 理解 Triton 的类型系统与类型转换
- 学会使用 `tl.constexpr` 进行编译时优化
- 为后续的 Softmax 和 GEMM 打下基础

In [1]:
import torch
import triton
import triton.language as tl

## 6.1 Triton 数学函数一览

| 函数 | 说明 | 典型用途 |
|------|------|----------|
| `tl.exp(x)` | e^x | Softmax |
| `tl.exp2(x)` | 2^x | 快速指数 |
| `tl.log(x)` | ln(x) | 交叉熵损失 |
| `tl.log2(x)` | log₂(x) | 信息论 |
| `tl.sqrt(x)` | √x | LayerNorm, Attention scaling |
| `tl.abs(x)` | \|x\| | L1 范数 |
| `tl.sigmoid(x)` | 1/(1+e^(-x)) | SiLU/Swish |
| `tl.sin(x)` / `tl.cos(x)` | 三角函数 | RoPE 位置编码 |
| `tl.maximum(a, b)` | max(a, b) | ReLU |
| `tl.minimum(a, b)` | min(a, b) | Clamp |
| `tl.dot(a, b)` | 矩阵乘 | GEMM (后面章节详讲) |

In [2]:
@triton.jit
def math_functions_kernel(
    x_ptr,
    out_exp_ptr, out_log_ptr, out_sqrt_ptr, out_sin_ptr,
    n, BLOCK_SIZE: tl.constexpr,
):
    """演示各种数学函数"""
    pid = tl.program_id(0)
    offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offsets < n
    
    x = tl.load(x_ptr + offsets, mask=mask, other=1.0)
    
    tl.store(out_exp_ptr + offsets, tl.exp(x), mask=mask)
    
    # log 需要正数输入，用 abs + epsilon 保护
    tl.store(out_log_ptr + offsets, tl.log(tl.abs(x) + 1e-6), mask=mask)
    
    # sqrt 需要非负输入
    tl.store(out_sqrt_ptr + offsets, tl.sqrt(tl.abs(x)), mask=mask)
    
    tl.store(out_sin_ptr + offsets, tl.sin(x), mask=mask)

n = 8
x = torch.tensor([0.0, 0.5, 1.0, 2.0, -1.0, 3.14, 0.1, -0.5], device='cuda')
outs = [torch.empty_like(x) for _ in range(4)]

math_functions_kernel[(1,)](x, *outs, n, BLOCK_SIZE=8)

print(f"x:      {x.tolist()}")
print(f"exp(x): {outs[0].tolist()}")
print(f"log|x|: {outs[1].tolist()}")
print(f"sqrt|x|:{outs[2].tolist()}")
print(f"sin(x): {outs[3].tolist()}")

x:      [0.0, 0.5, 1.0, 2.0, -1.0, 3.140000104904175, 0.10000000149011612, -0.5]
exp(x): [1.0, 1.6487212181091309, 2.7182817459106445, 7.389056205749512, 0.3678794503211975, 23.103872299194336, 1.1051709651947021, 0.6065306663513184]
log|x|: [-13.815510749816895, -0.6931451559066772, 9.536738616588991e-07, 0.6931476593017578, 9.536738616588991e-07, 1.1442230939865112, -2.30257511138916, -0.6931451559066772]
sqrt|x|:[0.0, 0.7071067690849304, 1.0, 1.4142135381698608, 1.0, 1.77200448513031, 0.3162277638912201, 0.7071067690849304]
sin(x): [0.0, 0.4794255495071411, 0.8414710164070129, 0.9092974662780762, -0.8414710164070129, 0.0015925479819998145, 0.0998334214091301, -0.4794255495071411]


## 6.2 类型系统

Triton 支持以下数据类型：

### 浮点类型
| Triton 类型 | 位宽 | PyTorch 类型 | 用途 |
|------------|------|-------------|------|
| `tl.float16` | 16 | `torch.float16` | 推理、GEMM 输入 |
| `tl.bfloat16` | 16 | `torch.bfloat16` | 训练（更大范围） |
| `tl.float32` | 32 | `torch.float32` | 累加器、精度要求高 |
| `tl.float64` | 64 | `torch.float64` | 科学计算（少用） |
| `tl.float8e5m2` | 8 | - | Hopper FP8 推理 |
| `tl.float8e4m3fn` | 8 | - | Hopper FP8 推理 |

### 整数类型
| Triton 类型 | 说明 |
|------------|------|
| `tl.int8` | 8位整数 |
| `tl.int16` | 16位整数 |
| `tl.int32` | 32位整数（默认） |
| `tl.int64` | 64位整数 |
| `tl.uint8` | 无符号8位 |

### FP16 vs BF16 vs FP32

```
类型      符号位  指数位  尾数位   范围           精度
FP32:     1       8      23      ±3.4e38       7位有效数字
FP16:     1       5      10      ±65504        3位有效数字
BF16:     1       8       7      ±3.4e38       2位有效数字
                  ↑               ↑
                  BF16 指数和      所以范围和 FP32 一样
                  FP32 一样大      但精度更低
```

**GEMM 的典型混合精度策略**：
- 输入: FP16/BF16 (节省带宽)
- 累加器: FP32 (保证精度)
- 输出: FP16/BF16 (节省带宽)

In [3]:
@triton.jit
def type_cast_kernel(
    x_fp32_ptr, out_fp16_ptr, out_bf16_ptr,
    n, BLOCK_SIZE: tl.constexpr,
):
    """
    类型转换示例：FP32 → FP16 / BF16
    使用 .to(dtype) 进行显式转换
    """
    pid = tl.program_id(0)
    offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offsets < n
    
    # 加载 FP32 数据
    x = tl.load(x_fp32_ptr + offsets, mask=mask)
    
    # 转换为 FP16 并存储
    x_fp16 = x.to(tl.float16)
    tl.store(out_fp16_ptr + offsets, x_fp16, mask=mask)
    
    # 转换为 BF16 并存储
    x_bf16 = x.to(tl.bfloat16)
    tl.store(out_bf16_ptr + offsets, x_bf16, mask=mask)

n = 8
x = torch.tensor([1.0, 0.1, 100.0, 0.001, 65504.0, 70000.0, 1e-8, 3.14], device='cuda')
out_fp16 = torch.empty(n, device='cuda', dtype=torch.float16)
out_bf16 = torch.empty(n, device='cuda', dtype=torch.bfloat16)

type_cast_kernel[(1,)](x, out_fp16, out_bf16, n, BLOCK_SIZE=8)

print("类型转换精度对比:")
print(f"{'FP32 原值':>15} {'FP16':>12} {'BF16':>12} {'FP16误差':>12} {'BF16误差':>12}")
for i in range(n):
    v = x[i].item()
    f16 = out_fp16[i].item()
    b16 = out_bf16[i].item()
    print(f"{v:15.6f} {f16:12.6f} {b16:12.6f} {abs(v-f16):12.6f} {abs(v-b16):12.6f}")

类型转换精度对比:
        FP32 原值         FP16         BF16       FP16误差       BF16误差
       1.000000     1.000000     1.000000     0.000000     0.000000
       0.100000     0.099976     0.100098     0.000024     0.000098
     100.000000   100.000000   100.000000     0.000000     0.000000
       0.001000     0.001000     0.000999     0.000000     0.000001
   65504.000000 65504.000000 65536.000000     0.000000    32.000000
   70000.000000          inf 70144.000000          inf   144.000000
       0.000000     0.000000     0.000000     0.000000     0.000000
       3.140000     3.140625     3.140625     0.000625     0.000625


## 6.3 `tl.constexpr` 详解

`tl.constexpr` 声明的参数在**编译时**就确定值（不是运行时）：

```python
@triton.jit
def my_kernel(
    ptr,             # 运行时参数（每次调用可以不同）
    N: tl.constexpr, # 编译时常量（不同的值会编译不同的 kernel）
):
    ...
```

### 为什么需要 constexpr？

1. **`tl.arange(0, N)`**: N 必须是编译时已知的
2. **`tl.zeros((M, N), ...)`**: 形状必须编译时确定
3. **`tl.static_range(N)`**: 循环展开需要
4. **编译器优化**: 知道确切值可以生成更优的代码

### 注意: 不同的 constexpr 值 = 不同的编译

```python
kernel[grid](ptr, BLOCK_SIZE=256)   # 编译 kernel 版本 A
kernel[grid](ptr, BLOCK_SIZE=512)   # 编译 kernel 版本 B (不同的二进制！)
kernel[grid](ptr, BLOCK_SIZE=256)   # 使用缓存的版本 A
```

In [4]:
@triton.jit
def constexpr_demo_kernel(
    x_ptr, out_ptr, n,
    BLOCK_SIZE: tl.constexpr,
    ACTIVATION: tl.constexpr,  # 字符串也可以是 constexpr！
):
    """
    constexpr 不只是数字，还可以是字符串！
    编译器会根据 ACTIVATION 的值生成不同版本的 kernel。
    """
    pid = tl.program_id(0)
    offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offsets < n
    x = tl.load(x_ptr + offsets, mask=mask)
    
    # 编译时分支! 只有一个分支会被编译
    if ACTIVATION == "relu":
        out = tl.maximum(x, 0.0)
    elif ACTIVATION == "sigmoid":
        out = tl.sigmoid(x)
    elif ACTIVATION == "tanh":
        # tanh(x) = 2 * sigmoid(2x) - 1
        out = 2.0 * tl.sigmoid(2.0 * x) - 1.0
    else:
        out = x  # identity
    
    tl.store(out_ptr + offsets, out, mask=mask)

n = 1000
x = torch.randn(n, device='cuda')

for act_name in ['relu', 'sigmoid', 'tanh', 'none']:
    out = torch.empty_like(x)
    grid = (triton.cdiv(n, 256),)
    constexpr_demo_kernel[grid](x, out, n, BLOCK_SIZE=256, ACTIVATION=act_name)
    
    # 验证
    if act_name == 'relu':
        expected = torch.relu(x)
    elif act_name == 'sigmoid':
        expected = torch.sigmoid(x)
    elif act_name == 'tanh':
        expected = torch.tanh(x)
    else:
        expected = x
    print(f"{act_name:>8}: 正确性={torch.allclose(out, expected, atol=1e-5)}")

    relu: 正确性=True
 sigmoid: 正确性=True
    tanh: 正确性=True
    none: 正确性=True


## 6.4 实战：RoPE 位置编码

Rotary Position Embedding (RoPE) 是现代 LLM 的标准位置编码，它使用 sin/cos：

In [5]:
@triton.jit
def rope_kernel(
    x_ptr, out_ptr,
    seq_len, head_dim,
    stride_s, stride_d,
    BLOCK_D: tl.constexpr,
):
    """
    简化版 RoPE: 对位置 pos 的嵌入向量应用旋转。
    
    公式: 对每对 (x[2i], x[2i+1]):
      θ_i = pos / 10000^(2i/d)
      x'[2i]   = x[2i] * cos(θ_i) - x[2i+1] * sin(θ_i)
      x'[2i+1] = x[2i] * sin(θ_i) + x[2i+1] * cos(θ_i)
    """
    pos = tl.program_id(0)  # 序列位置
    
    # 生成维度索引 [0, 2, 4, ..., d-2]
    pair_offsets = tl.arange(0, BLOCK_D // 2)  # 对的索引
    mask = pair_offsets < head_dim // 2
    
    # 计算频率: θ_i = pos / 10000^(2i/d)
    # 等价于: θ_i = pos * exp(-2i/d * log(10000))
    freq = pos * tl.exp(-2.0 * pair_offsets.to(tl.float32) / head_dim * tl.log(10000.0))
    cos_val = tl.cos(freq)
    sin_val = tl.sin(freq)
    
    # 加载偶数和奇数位置的值
    even_offsets = pos * stride_s + (pair_offsets * 2) * stride_d
    odd_offsets = pos * stride_s + (pair_offsets * 2 + 1) * stride_d
    
    x_even = tl.load(x_ptr + even_offsets, mask=mask)
    x_odd = tl.load(x_ptr + odd_offsets, mask=mask)
    
    # 旋转
    out_even = x_even * cos_val - x_odd * sin_val
    out_odd = x_even * sin_val + x_odd * cos_val
    
    tl.store(out_ptr + even_offsets, out_even, mask=mask)
    tl.store(out_ptr + odd_offsets, out_odd, mask=mask)

# 测试
seq_len, head_dim = 16, 64
x = torch.randn(seq_len, head_dim, device='cuda')
out = torch.empty_like(x)

BLOCK_D = triton.next_power_of_2(head_dim)
rope_kernel[(seq_len,)](
    x, out, seq_len, head_dim,
    x.stride(0), x.stride(1),
    BLOCK_D=BLOCK_D,
)

print(f"RoPE 输入形状: {x.shape}")
print(f"RoPE 输出形状: {out.shape}")
print(f"输出包含 NaN: {out.isnan().any().item()}")
print(f"输出范围: [{out.min().item():.4f}, {out.max().item():.4f}]")

RoPE 输入形状: torch.Size([16, 64])
RoPE 输出形状: torch.Size([16, 64])
输出包含 NaN: False
输出范围: [-3.4121, 3.3878]


## 6.5 `tl.dot` 预览

`tl.dot(a, b)` 是 Triton 中最重要的函数之一，它执行**块级矩阵乘法**并自动映射到 **Tensor Core**：

```python
# a: (BLOCK_M, BLOCK_K)  fp16/bf16
# b: (BLOCK_K, BLOCK_N)  fp16/bf16
# c: (BLOCK_M, BLOCK_N)  fp32 (累加器)
c = tl.dot(a, b)          # c += a @ b
c = tl.dot(a, b, acc=c)   # 累加到已有的 c
```

**这等价于 CUDA 中的 `wmma::mma_sync` 或 `mma.sync` PTX 指令！**

但在 Triton 中只需要一行代码，编译器会自动：
- 选择合适的 Tensor Core 指令
- 优化数据布局
- 处理 warp 级别的协作

我们会在第09章开始详细使用 `tl.dot`。

In [6]:
@triton.jit
def dot_preview_kernel(a_ptr, b_ptr, c_ptr, BLOCK: tl.constexpr):
    """tl.dot 快速预览"""
    offs = tl.arange(0, BLOCK)
    
    # 加载两个小矩阵 (BLOCK x BLOCK)
    a = tl.load(a_ptr + offs[:, None] * BLOCK + offs[None, :])  # (BLOCK, BLOCK)
    b = tl.load(b_ptr + offs[:, None] * BLOCK + offs[None, :])  # (BLOCK, BLOCK)
    
    # 矩阵乘！就这么简单！
    c = tl.dot(a, b)  # (BLOCK, BLOCK)
    
    tl.store(c_ptr + offs[:, None] * BLOCK + offs[None, :], c)

BLOCK = 16
a = torch.randn(BLOCK, BLOCK, device='cuda', dtype=torch.float16)
b = torch.randn(BLOCK, BLOCK, device='cuda', dtype=torch.float16)
c = torch.empty(BLOCK, BLOCK, device='cuda', dtype=torch.float32)

dot_preview_kernel[(1,)](a, b, c, BLOCK=BLOCK)

expected = (a.float() @ b.float())
print(f"tl.dot 矩阵乘预览:")
print(f"  形状: ({BLOCK},{BLOCK}) @ ({BLOCK},{BLOCK}) = ({BLOCK},{BLOCK})")
print(f"  正确性: {torch.allclose(c, expected, atol=1e-2)}")
print(f"  最大误差: {(c - expected).abs().max().item():.6f}")

tl.dot 矩阵乘预览:
  形状: (16,16) @ (16,16) = (16,16)
  正确性: True
  最大误差: 0.000001


## Part 1 总结

恭喜！你已经完成了 Triton 基础部分的全部 6 章。回顾一下学到的核心概念：

```
第01章: @triton.jit, tl.program_id, tl.load/store, kernel 启动
第02章: 多维 grid, stride, lambda grid
第03章: 指针运算, mask, make_block_ptr, advance
第04章: tl.arange, 广播, 2D 索引模式
第05章: tl.where, tl.sum/max/min, atomic 操作, 循环
第06章: 数学函数, 类型转换, constexpr, tl.dot 预览
```

### 你现在掌握的能力
- 编写和启动 Triton kernel
- 处理 1D 和 2D 数据
- 实现各种元素操作和归约
- 理解 GPU 内存访问模式

### 接下来
Part 2 将把这些基础组合起来，实现真正有用的 kernel：
- **Softmax**: 归约 + 数学函数的综合应用
- **矩阵乘法**: tl.dot + 分块 + 循环的终极结合
- **算子融合**: 减少内存访问的关键技术
- **自动调优**: 找到最优的 kernel 配置